In [ ]:
!pip install yfinance pandas tabulate tqdm -q


In [ ]:
import yfinance as yf
import pandas as pd
import time
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

def get_metrics(ticker):
    try:
        stock = yf.Ticker(ticker.strip().upper())
        info = stock.info
        if not info or not info.get('regularMarketPrice'):
            return {'Ticker': ticker, 'Error': 'No data'}
        def safe(v, div=1):
            try: return round(v / div, 2) if v else None
            except: return None
        return {
            'Ticker': ticker.upper(),
            'Name': info.get('longName') or info.get('shortName'),
            'Sector': info.get('sector'),
            'Price': info.get('regularMarketPrice'),
            'MarketCap_mln': safe(info.get('marketCap'), 1e6),
            'EV_mln': safe(info.get('enterpriseValue'), 1e6),
            'PE_trailing': info.get('trailingPE'),
            'PE_forward': info.get('forwardPE'),
            'PEG': info.get('pegRatio'),
            'PB': info.get('priceToBook'),
            'PS': info.get('priceToSalesTrailing12Months'),
            'EV_EBITDA': info.get('enterpriseToEbitda'),
            'EV_Rev': info.get('enterpriseToRevenue'),
            'ROE': safe(info.get('returnOnEquity'), 0.01),
            'NetMargin': safe(info.get('profitMargins'), 0.01),
            'RevGrowth': safe(info.get('revenueGrowth'), 0.01),
            'DebtEquity': info.get('debtToEquity')
        }
    except:
        return {'Ticker': ticker, 'Error': 'Failed'}

def analyze_large_list(tickers, batch=20):
    print(f'Processing {len(tickers)} tickers in batches of {batch}...')
    results = []
    for i in tqdm(range(0, len(tickers), batch)):
        batch_tickers = tickers[i:i+batch]
        for t in batch_tickers:
            results.append(get_metrics(t))
        time.sleep(1.2)
    df = pd.DataFrame(results)
    if 'Error' in df.columns:
        errors = df[df['Error'].notna()]['Ticker'].tolist()
        if errors: print('Failed tickers:', errors)
        df = df[df['Error'].isna()].drop(columns=['Error'], errors='ignore')
    if df.empty: return df
    # vs median columns
    for col in ['PE_trailing', 'PE_forward', 'PEG', 'PB', 'PS', 'EV_EBITDA', 'EV_Rev']:
        if col in df.columns:
            med = df[col].median()
            if med and med > 0:
                df[col + '_vsMed'] = (df[col] / med).round(2)
    return df

# ==================== PASTE YOUR TICKERS HERE ====================
tickers = ['MSFT','GOOGL','AMZN','META','AAPL','NVDA','TSLA','AMD']  # change this list
# ============================================================

df = analyze_large_list(tickers)
if not df.empty:
    print('
' + '='*90)
    print('VALUATION TABLE')
    print('='*90)
    print(df.to_markdown(index=False))

    prompt = f'''You are a strict value investor. Analyze this table.

Focus on which stocks are cheap vs the group median (vsMed < 1 = cheaper).

Data:
{df.to_markdown(index=False)}
'''
    print('
' + '='*90)
    print('COPY THIS PROMPT TO GROK:')
    print('='*90)
    print(prompt)
